# 01 EDA - Redrob Intelligent Candidate Discovery & Ranking Challenge
This notebook parses the raw datasets and documentation.

In [ ]:
%pip install pandas numpy tqdm matplotlib seaborn python-docx python-dateutil

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from docx import Document
from dateutil import parser

# 2. Set base_dir
base_dir = "/content/ai-candidate-ranking"
# Fallback if running locally instead of Colab
if not os.path.exists(base_dir):
    base_dir = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."

raw_dir = os.path.join(base_dir, "data", "raw")

### 3. Parse Job Description
Extract plain text and key sections.

In [ ]:
def extract_docx_text(file_path):
    try:
        doc = Document(file_path)
        return "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
    except Exception as e:
        return f"Error loading {file_path}: {e}"

jd_path = os.path.join(raw_dir, "job_description.docx")
jd_text = extract_docx_text(jd_path)

print("=== Job Description ===")
# Printing the full text so you can inspect the key sections.
# The text contains details on what the role needs, absolute necessities, disqualifiers, and hackathon notes.
print(jd_text)


### 3. Parse Submission Spec
Extract required columns, limits, and rules.

In [ ]:
spec_path = os.path.join(raw_dir, "submission_spec.docx")
spec_text = extract_docx_text(spec_path)

print("=== Submission Specification ===")
print(spec_text)


### 3. Parse Redrob Signals Doc
Extract the 23 redrob_signals and their meanings.

In [ ]:
signals_path = os.path.join(raw_dir, "redrob_signals_doc.docx")
print("=== Redrob Signals ===")
try:
    doc = Document(signals_path)
    # Often signals are presented in tables in the docx
    for table in doc.tables:
        for row in table.rows:
            print(" | ".join([cell.text.strip() for cell in row.cells]))
        print("-" * 50)
    # Fallback if they are in normal text
    print("--- Text Content ---")
    print(extract_docx_text(signals_path))
except Exception as e:
    print(f"Error loading {signals_path}: {e}")


### 3. Parse Candidate Schema
Load JSON and pretty-print the schema for profile, career_history, skills, and redrob_signals.

In [ ]:
schema_path = os.path.join(raw_dir, "candidate_schema.json")
try:
    with open(schema_path, "r") as f:
        schema = json.load(f)
    print("=== Candidate Schema ===")
    
    # Pretty print specific sections if they exist
    for key in ["profile", "career_history", "skills", "redrob_signals"]:
        if key in schema:
            print(f"\n--- {key.upper()} ---")
            print(json.dumps(schema[key], indent=2))
        else:
            # If schema format is different, just print it all
            pass
    
    print("\n--- Full Schema ---")
    print(json.dumps(schema, indent=2))
except Exception as e:
    print(f"Error loading {schema_path}: {e}")


### 4. Load Sample Candidates
Load and format candidates, extracting specific fields.

In [ ]:
sample_path = os.path.join(raw_dir, "sample_candidates.json")
sample_candidates = []
try:
    with open(sample_path, "r") as f:
        content = f.read().strip()
        if content:
            sample_candidates = json.loads(content)
        else:
            print(f"{sample_path} is empty.")
            
    if sample_candidates:
        print(f"Loaded {len(sample_candidates)} candidates.\n")
        
        print("=== Top 3 Candidates (Full Formatting) ===")
        print(json.dumps(sample_candidates[:3], indent=2))
        
        print("\n=== Candidate Summaries ===")
        for cand in sample_candidates:
            cid = cand.get("candidate_id", "N/A")
            profile = cand.get("profile", {})
            skills = cand.get("skills", [])
            signals = cand.get("redrob_signals", {})
            
            print(f"\nCandidate ID: {cid}")
            print(f"  Headline: {profile.get('headline')}")
            print(f"  Summary: {profile.get('summary')}")
            print(f"  Location: {profile.get('location')}")
            print(f"  Years of Exp: {profile.get('years_of_experience')}")
            print(f"  Current Title: {profile.get('current_title')}")
            print(f"  Current Industry: {profile.get('current_industry')}")
            
            print(f"  Skills Count: {len(skills)}")
            if skills:
                # Extract skill names whether they are strings or dicts
                skill_names = [s.get('name', str(s)) if isinstance(s, dict) else str(s) for s in skills]
                print(f"  Example Skills (up to 10): {', '.join(skill_names[:10])}")
            
            print("  Key Redrob Signals:")
            print(f"    - profile_completeness_score: {signals.get('profile_completeness_score')}")
            print(f"    - last_active_date: {signals.get('last_active_date')}")
            print(f"    - open_to_work_flag: {signals.get('open_to_work_flag')}")
            print(f"    - recruiter_response_rate: {signals.get('recruiter_response_rate')}")
            print(f"    - interview_completion_rate: {signals.get('interview_completion_rate')}")
            print(f"    - github_activity_score: {signals.get('github_activity_score')}")
            
except Exception as e:
    print(f"Error processing {sample_path}: {e}")


### 5. Flatten Candidates to DataFrame
Extract key profile and signal features for tabular analysis. Identify AI core skills based on keywords.

In [ ]:
import re
from IPython.display import display

ai_keywords = [
    "machine learning", "deep learning", "nlp", "search", "retrieval", 
    "embedding", "vector", "pytorch", "tensorflow", "mlflow", "llm", "ai",
    "artificial intelligence", "recommender", "recommendation"
]
ai_pattern = re.compile("|".join(ai_keywords), re.IGNORECASE)

records = []
for cand in sample_candidates:
    cid = cand.get("candidate_id")
    profile = cand.get("profile", {})
    signals = cand.get("redrob_signals", {})
    skills = cand.get("skills", [])
    
    # Extract AI skills
    ai_skills_count = 0
    for s in skills:
        s_name = s.get("name", str(s)) if isinstance(s, dict) else str(s)
        if ai_pattern.search(s_name):
            ai_skills_count += 1
            
    records.append({
        "candidate_id": cid,
        "years_of_experience": profile.get("years_of_experience"),
        "current_title": profile.get("current_title"),
        "current_company": profile.get("current_company"),
        "current_industry": profile.get("current_industry"),
        "total_skills_count": len(skills),
        "ai_core_skills_count": ai_skills_count,
        "recruiter_response_rate": signals.get("recruiter_response_rate"),
        "profile_completeness_score": signals.get("profile_completeness_score"),
        "interview_completion_rate": signals.get("interview_completion_rate"),
        "github_activity_score": signals.get("github_activity_score")
    })

df = pd.DataFrame(records)
numeric_cols = ["years_of_experience", "ai_core_skills_count", "recruiter_response_rate", 
               "profile_completeness_score", "interview_completion_rate", "github_activity_score"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

display(df.head())

### 6. Plot Distributions
Visualize the distributions of experience, AI skills, and key Redrob signals.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df["years_of_experience"].dropna(), kde=True, ax=axes[0,0], color="skyblue")
axes[0,0].set_title("Years of Experience Distribution")

sns.histplot(df["ai_core_skills_count"].dropna(), kde=False, ax=axes[0,1], color="salmon", bins=15)
axes[0,1].set_title("AI Core Skills Count Distribution")

sns.histplot(df["recruiter_response_rate"].dropna(), kde=True, ax=axes[1,0], color="lightgreen")
axes[1,0].set_title("Recruiter Response Rate Distribution")

sns.histplot(df["profile_completeness_score"].dropna(), kde=True, ax=axes[1,1], color="gold")
axes[1,1].set_title("Profile Completeness Score Distribution")

plt.tight_layout()
plt.show()

### 7. Top Candidates Analysis
Show top 15 candidates by AI core skills count, and prioritize high response rates among those with AI-relevant titles.

In [ ]:
# 1. Top 15 by ai_core_skills_count
top_by_skills = df.sort_values(by="ai_core_skills_count", ascending=False).head(15)
print("=== Top 15 by AI Core Skills Count ===")
display(top_by_skills[["candidate_id", "current_title", "ai_core_skills_count", "total_skills_count", "recruiter_response_rate"]])

# 2. Top 15 by recruiter response rate among candidates with AI titles
ai_titles = ["ml engineer", "machine learning", "data scientist", "ai engineer", 
             "search engineer", "recommendation", "applied scientist"]
ai_title_pattern = re.compile("|".join(ai_titles), re.IGNORECASE)

df["has_ai_title"] = df["current_title"].fillna("").apply(lambda x: bool(ai_title_pattern.search(x)))
ai_title_candidates = df[df["has_ai_title"]]
top_by_response = ai_title_candidates.sort_values(by=["recruiter_response_rate", "ai_core_skills_count"], ascending=[False, False]).head(15)

print("\n=== Top 15 AI-Titled Candidates by Recruiter Response Rate ===")
display(top_by_response[["candidate_id", "current_title", "ai_core_skills_count", "recruiter_response_rate"]])

### 8. Trap/Honeypot Identification (Keyword Stuffing)
Identify candidates with completely unrelated titles (e.g., "Software Engineer - Frontend" or "HR Manager") but extremely high AI core skills counts. These are likely keyword stuffers or honeypots trying to game naive search algorithms.

In [ ]:
# Mismatch: NOT an AI title, but very high ai_core_skills_count
mismatch_candidates = df[~df["has_ai_title"]].copy()

# Sort by AI skills descending
obvious_mismatches = mismatch_candidates.sort_values(by="ai_core_skills_count", ascending=False).head(10)

print("=== 10 Obvious Mismatches (Potential Keyword Stuffers) ===")
display(obvious_mismatches[["candidate_id", "current_title", "ai_core_skills_count", "total_skills_count", "recruiter_response_rate"]])

### Summary: "Good Fit" vs "Trap"

Based on our EDA and the Job Description, here is how we can distinguish genuine candidates from honeypots/stuffers:

**The "Good Fit" Candidate**:
* **Relevant Experience**: Has a strong overlap between their `current_title` (e.g., "Machine Learning Engineer", "Search Engineer") and the required AI core skills.
* **Balanced Skills**: Possesses a realistic ratio of `ai_core_skills_count` to `total_skills_count`. (e.g., They don't have 500 skills where 200 are AI-related but their title is "Frontend Developer").
* **Strong Redrob Signals**: High `recruiter_response_rate` (meaning they actually engage), high `interview_completion_rate`, and recent `last_active_date`.
* **JD Alignment**: Actually meets the specific structural requirements (like years of experience) mentioned in the job description without throwing red flags.

**The "Trap" (Honeypot or Keyword Stuffer)**:
* **Title/Skill Mismatch**: An obviously unrelated title ("Backend Engineer" or "Marketing") paired with an absurdly high `ai_core_skills_count`. They just paste JD keywords into their resume.
* **Suspicious Signals**: Extremely low `recruiter_response_rate` (they spam resumes but never reply to recruiters), or a `profile_completeness_score` that is extremely high in text fields but missing fundamental verified signals. 
* **The "Everything" Candidate**: A `total_skills_count` that is impossibly high, aiming to hit every possible search query.

By penalizing these "Trap" patterns using the Redrob signals and semantic mismatch detection (e.g., Title vs Skills embeddings), our ranking pipeline can effectively filter out honeypots.